# CS-781 Capstone — Phase 3: Rhetorical Role Prediction (RR)

This notebook fine-tunes **your own Phase-1 language models** (A, B, C, D) to label every sentence of a legal judgment with its rhetorical role (13 roles), for the Corporate Law (CL) and Information Technology (IT) domains, and writes one `submission.zip` per experiment.

## Before you run it
1. **Settings (right panel):** Accelerator **GPU T4 x2**, Internet **on**.
2. **Add-ons → Secrets:** turn on the secret **`HF`** (your Hugging Face token) for this notebook. The data is downloaded from `Exploration-Lab/CS-781-Capstone/Phase_3_RR`.
3. **Add Input:** your Phase-1 dataset (the same one as in Phase 2). Section 6 prints which file is used for A, B, C and D.
4. *Save Version → Save & Run All.* To continue later, attach this notebook's previous output too: finished experiments are loaded, not retrained.

## The model
Each sentence is read by the Phase-1 model and turned into one vector (learned average over all layers, mean over the sentence's tokens). A **BiLSTM over the sequence of sentence vectors** gives every sentence its document context, a learned **CL / IT vector** tells the model the domain, and a **CRF** picks the most likely sequence of roles (roles follow the structure of a judgment: facts → arguments → precedents → ratio → ruling).

| Model | Phase-1 notebook | Design |
|---|---|---|
| A | `phase1-a` | GPT decoder, first run (220M tokens) |
| B | `phase1-b` | GQA + RoPE + RMSNorm + SwiGLU decoder |
| C | `phase1-c` | encoder-decoder (its encoder is used, plus its token embeddings: the encoder output does not depend on the words) |
| D | `phase1-d-optimisedA` | GPT decoder, optimised run (400M tokens) |

| Submission | Model | Technique |
|---|---|---|
| R1_D_lorma | D | **LoRMA** (compulsory) |
| R2_B_rslora | B | **rsLoRA** (rank-stabilised LoRA) |
| R3_C_ia3 | C | **IA³** (learned rescaling of keys, values and feed-forward activations) |
| R4_A4_kd | A (first 4 layers) | **Knowledge distillation**, teacher R1 |

Extra experiments for the analysis (Section 2): sentence-only vs document context, full fine-tuning vs LoRMA, class-balanced loss, KD without the teacher, CL-only / IT-only training, LoRA vs LoRMA.

## Rules this follows (course document §9)
Only your Phase-1 models are used as starting points; no other pretrained model, weights or API. Training uses only the provided RR data. Exactly one role per test sentence, in the original order, with the official label names — checked before every `submission.zip` is written.

## 1. Setup and Hugging Face login

In [1]:
import os
import re
import sys
import json
import math
import time
import glob
import copy
import random
import shutil
import zipfile
import hashlib
import warnings
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF")
except Exception as e:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    print("Kaggle secret 'HF' not available (" + type(e).__name__ + "); using the HF_TOKEN environment variable if set.")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("Hugging Face login successful.")
    except Exception as e:
        print("Hugging Face login failed:", repr(e))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("PyTorch:", torch.__version__, "| device:", DEVICE)
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    print("WARNING: no GPU. Use Kaggle 'GPU T4 x2' for real runs.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.
PyTorch: 2.11.0+cu128 | device: cuda
GPU 0: Tesla T4
GPU 1: Tesla T4


## 2. Configuration and experiments

Each experiment is one line: which Phase-1 model, which technique, which head. `RUN_EXPERIMENTS` chooses what runs.

In [ ]:
SEED = 42
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
PHASE3_DIR = "Phase_3_RR"
DOMAINS = ["CL", "IT"]

INPUT_ROOT = os.environ.get("CS781_INPUT_ROOT", "/kaggle/input")
WORK_ROOT = os.environ.get("CS781_P3_WORK_ROOT",
                           "/kaggle/working/cs781_phase3" if os.path.isdir("/kaggle/working") else os.path.abspath("cs781_phase3"))
LOCAL_DATA_DIR = os.environ.get("CS781_P3_DATA_DIR", "")
os.makedirs(WORK_ROOT, exist_ok=True)

PHASE1_MODELS = {
    "A": dict(architecture="A", pick="other"),
    "B": dict(architecture="B", pick="best"),
    "C": dict(architecture="C", pick="best"),
    "D": dict(architecture="A", pick="best"),
}

DEFAULTS = dict(
    backbone="D",
    n_layers=None,
    technique="lorma",
    context="bilstm",
    decoder="crf",
    domain_embedding=True,
    train_domains=["CL", "IT"],
    max_sent_tokens=96,
    chunk_sentences=32,
    chunks_per_batch=4,
    eval_sentence_batch=128,
    epochs=12, patience=4,
    lr_backbone=3e-5,
    lr_peft=5e-4,
    lr_head=1e-3,
    weight_decay=0.01, warmup_frac=0.1, grad_clip=1.0,
    dropout=0.1, head_dropout=0.3,
    lstm_hidden=256, lstm_layers=1,
    lora_r=16, lora_alpha=32,
    lora_scaling="standard",
    peft_targets="qv",
    lorma_variant="plus",
    adapter_dim=64,
    embedding_state=False,
    class_weighting=False,
    teacher=None, kd_temperature=2.0, kd_weight=0.5,
    seed=SEED,
)

EXPERIMENTS = {
    "R1_D_lorma":   dict(backbone="D", technique="lorma"),
    "R2_B_rslora":  dict(backbone="B", technique="lora", lora_scaling="rs"),
    "R3_C_ia3":     dict(backbone="C", technique="ia3", embedding_state=True, lr_peft=3e-3),
    "R4_A4_kd":     dict(backbone="A", n_layers=4, technique="full", lstm_hidden=128,
                         lr_backbone=5e-5, teacher="R1_D_lorma"),
    "X1_D_lorma_sentence_only": dict(backbone="D", technique="lorma", context="none"),
    "X2_D_full":                dict(backbone="D", technique="full"),
    "X3_D_lorma_balanced":      dict(backbone="D", technique="lorma", decoder="softmax", class_weighting=True),
    "X4_A4_no_kd":              dict(backbone="A", n_layers=4, technique="full", lstm_hidden=128, lr_backbone=5e-5),
    "X5_D_lorma_CL_only":       dict(backbone="D", technique="lorma", train_domains=["CL"]),
    "X6_D_lorma_IT_only":       dict(backbone="D", technique="lorma", train_domains=["IT"]),
    "X7_D_lora":                dict(backbone="D", technique="lora"),
    "X8_C_full":                dict(backbone="C", technique="full", embedding_state=True),
    "X9_D_lorma_balanced_all":  dict(backbone="D", technique="lorma", decoder="softmax", class_weighting=True, final_mode="retrain_all"),
}
RUN_EXPERIMENTS = ["R1_D_lorma", "R2_B_rslora", "R3_C_ia3", "R4_A4_kd", "X8_C_full", "X2_D_full", "X3_D_lorma_balanced", "X9_D_lorma_balanced_all"]
if os.environ.get("CS781_P3_RUN"):
    RUN_EXPERIMENTS = [x for x in os.environ["CS781_P3_RUN"].split(",") if x]

FINAL_MODE = "retrain_all"

SMOKE_TEST = os.environ.get("CS781_P3_SMOKE", "0") == "1"
if SMOKE_TEST:
    DEFAULTS.update(epochs=2, patience=2, max_sent_tokens=24, chunk_sentences=8, chunks_per_batch=2, lstm_hidden=16,
                    lora_r=4, adapter_dim=8)
    for _cfg in EXPERIMENTS.values():
        if _cfg.get("n_layers"):
            _cfg["n_layers"] = 1
        _cfg.pop("lstm_hidden", None)
    print("*** SMOKE TEST MODE: tiny settings, results are meaningless ***")

def experiment_config(name):
    cfg = dict(DEFAULTS)
    cfg.update(EXPERIMENTS[name])
    cfg["name"] = name
    return cfg

def restore_previous_outputs():
    """Copy a previous run of this notebook (attached with "Add Input") into the working folder, so finished
    experiments are re-used instead of trained again."""
    if not os.path.isdir(INPUT_ROOT):
        return
    copied = 0
    for dirpath, dirnames, _ in os.walk(INPUT_ROOT):
        if os.path.basename(dirpath) == "cs781_phase3":
            for sub, _, files in os.walk(dirpath):
                for name in files:
                    src = os.path.join(sub, name)
                    dst = os.path.join(WORK_ROOT, os.path.relpath(src, dirpath))
                    if not os.path.exists(dst):
                        os.makedirs(os.path.dirname(dst), exist_ok=True)
                        shutil.copy2(src, dst)
                        copied += 1
            dirnames[:] = []
        elif dirpath[len(INPUT_ROOT):].count(os.sep) >= 6:
            dirnames[:] = []
    if copied:
        print(f"Restored {copied} file(s) from a previous Phase-3 run.")

restore_previous_outputs()
print("Working folder:", WORK_ROOT)
print("Experiments to run:", RUN_EXPERIMENTS)
for _name in RUN_EXPERIMENTS:
    _c = experiment_config(_name)
    print(f"  {_name:26s} model {_c['backbone']}{'' if not _c['n_layers'] else ' (first ' + str(_c['n_layers']) + ' layers)'} | "
          f"{_c['technique']:6s} | context {_c['context']} | {_c['decoder']}" + (f" | teacher {_c['teacher']}" if _c['teacher'] else ""))

Restored 26 file(s) from a previous Phase-3 run.
Working folder: /kaggle/working/cs781_phase3
Experiments to run: ['R1_D_lorma', 'R2_B_rslora', 'R3_C_ia3', 'R4_A4_kd', 'X8_C_full', 'X2_D_full', 'X3_D_lorma_balanced']
  R1_D_lorma                 model D | lorma  | context bilstm | crf
  R2_B_rslora                model B | lora   | context bilstm | crf
  R3_C_ia3                   model C | ia3    | context bilstm | crf
  R4_A4_kd                   model A (first 4 layers) | full   | context bilstm | crf | teacher R1_D_lorma
  X8_C_full                  model C | full   | context bilstm | crf
  X2_D_full                  model D | full   | context bilstm | crf
  X3_D_lorma_balanced        model D | lorma  | context bilstm | softmax


## 3. Load the RR data

In [3]:
def phase3_file(relative):
    """Local path of a Phase-3 file (attached dataset, local folder, or Hugging Face download)."""
    candidates = []
    if LOCAL_DATA_DIR:
        candidates.append(os.path.join(LOCAL_DATA_DIR, PHASE3_DIR, relative))
        candidates.append(os.path.join(LOCAL_DATA_DIR, relative))
    if os.path.isdir(INPUT_ROOT):
        for depth in range(1, 5):
            candidates += sorted(glob.glob(os.path.join(INPUT_ROOT, *(["*"] * depth), PHASE3_DIR, relative)))
    for c in candidates:
        if os.path.isfile(c):
            return c
    from huggingface_hub import hf_hub_download
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=f"{PHASE3_DIR}/{relative}", repo_type="dataset", token=HF_TOKEN)

def load_json_file(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def write_json(obj, path):
    with open(path + ".tmp", "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    os.replace(path + ".tmp", path)

_vocab = load_json_file(phase3_file("Train/label_vocab.json"))
if isinstance(_vocab, dict):
    LABELS = [lab for lab, _ in sorted(_vocab.items(), key=lambda kv: int(kv[1]))]
else:
    LABELS = [str(x) for x in _vocab]
LABEL_TO_ID = {lab: i for i, lab in enumerate(LABELS)}
NUM_LABELS = len(LABELS)

def read_documents(relative, has_labels):
    """-> list of (doc_id, [sentences], [label ids] or None). Train/dev files are a list of {id, text, labels};
    test files map doc_id -> [sentences] (a list of {id, text} is accepted too)."""
    data = load_json_file(phase3_file(relative))
    items = list(data.items()) if isinstance(data, dict) else [(d.get("id"), d) for d in data]
    out = []
    for i, (doc_id, doc) in enumerate(items):
        if isinstance(doc, dict):
            doc_id = str(doc.get("id", doc_id if doc_id is not None else i))
            text, labels = doc.get("text", doc.get("sentences")), doc.get("labels")
        else:
            text, labels = doc, None
        text = ["" if s is None else str(s) for s in text]
        if has_labels:
            if labels is None or len(labels) != len(text):
                raise ValueError(f"{relative}: document {doc_id} has {len(text)} sentences but "
                                 f"{None if labels is None else len(labels)} labels")
            unknown = sorted({str(l) for l in labels} - set(LABEL_TO_ID))
            if unknown:
                raise ValueError(f"{relative}: unknown labels {unknown}")
            labels = [LABEL_TO_ID[str(l)] for l in labels]
        else:
            labels = None
        out.append((str(doc_id), text, labels))
    return out

DOCS, SPLIT_KEYS = {}, {"train": [], "dev": [], "test": []}
for _domain in DOMAINS:
    for _split, _file, _labelled in [("train", f"Train/{_domain}_train.json", True), ("dev", f"Train/{_domain}_dev.json", True),
                                     ("test", f"Test/{_domain}_test.json", False)]:
        for _doc_id, _text, _labels in read_documents(_file, _labelled):
            _key = (_split, _domain, _doc_id)
            DOCS[_key] = {"text": _text, "labels": _labels, "domain": _domain}
            SPLIT_KEYS[_split].append(_key)
_test_ids = [k[2] for k in SPLIT_KEYS["test"]]
assert len(set(_test_ids)) == len(_test_ids), "a test document id appears in both domains - prediction.json would clash"

print(f"{NUM_LABELS} labels: {LABELS}")
for _split in ("train", "dev", "test"):
    for _domain in DOMAINS:
        _keys = [k for k in SPLIT_KEYS[_split] if k[1] == _domain]
        _n = [len(DOCS[k]["text"]) for k in _keys]
        _w = [len(s.split()) for k in _keys for s in DOCS[k]["text"]]
        print(f"{_split:5s} {_domain}: {len(_keys):3d} documents | {sum(_n):6,} sentences (max {max(_n)} per document) | "
              f"words per sentence: mean {np.mean(_w):.1f}, max {max(_w)}")
LABEL_COUNTS = {split: Counter(LABELS[l] for k in SPLIT_KEYS[split] for l in DOCS[k]["labels"]) for split in ("train", "dev")}
print("\nRole counts:")
print(f"{'role':24s} {'train CL':>9s} {'train IT':>9s} {'dev CL':>7s} {'dev IT':>7s}")
for _lab in LABELS:
    _c = [sum(DOCS[k]["labels"].count(LABEL_TO_ID[_lab]) for k in SPLIT_KEYS[s] if k[1] == d)
          for s in ("train", "dev") for d in DOMAINS]
    print(f"{_lab:24s} {_c[0]:9,} {_c[1]:9,} {_c[2]:7,} {_c[3]:7,}")

label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

13 labels: ['Fact', 'Issue', 'ArgumentPetitioner', 'ArgumentRespondent', 'Statute', 'Dissent', 'PrecedentReliedUpon', 'PrecedentNotReliedUpon', 'PrecedentOverruled', 'RulingByLowerCourt', 'RatioOfTheDecision', 'RulingByPresentCourt', 'None']
train CL:  40 documents | 10,593 sentences (max 1816 per document) | words per sentence: mean 32.7, max 733
train IT:  40 documents |  6,252 sentences (max 442 per document) | words per sentence: mean 35.3, max 1054
dev   CL:   5 documents |  1,396 sentences (max 848 per document) | words per sentence: mean 33.7, max 339
dev   IT:   5 documents |    746 sentences (max 382 per document) | words per sentence: mean 32.3, max 287
test  CL:   5 documents |  1,339 sentences (max 942 per document) | words per sentence: mean 33.2, max 533
test  IT:   5 documents |    858 sentences (max 420 per document) | words per sentence: mean 37.6, max 316

Role counts:
role                      train CL  train IT  dev CL  dev IT
Fact                         2,979     

## 4. Phase-1 tokenizer

Sentences are encoded exactly as in Phase 1: every word is BPE-encoded and followed by `</w>`.

In [4]:
SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>"]
PAD_ID = 0

class BPETokenizer:
    """Loads the byte-level BPE tokenizer saved by a Phase-1 notebook (tokenizer.json) and encodes text with the
    same merge rules. Only loading and encoding are needed here; the tokenizer is NOT retrained."""

    def __init__(self, directory):
        with open(os.path.join(directory, "tokenizer.json"), "r", encoding="utf-8") as f:
            data = json.load(f)
        self.token_to_id = {str(k): int(v) for k, v in data["token_to_id"].items()}
        self.merge_ranks = {tuple(pair): i for i, pair in enumerate(data["merges"])}
        self.unk_id = self.token_to_id["<unk>"]
        self.eow_id = self.token_to_id["</w>"]
        self.vocab_size = len(self.token_to_id)
        self._cache = {}

    def _merge(self, text):
        tokens = [str(b) for b in text.encode("utf-8")]
        while len(tokens) > 1:
            best_rank, best_index = None, -1
            for i in range(len(tokens) - 1):
                rank = self.merge_ranks.get((tokens[i], tokens[i + 1]))
                if rank is not None and (best_rank is None or rank < best_rank):
                    best_rank, best_index = rank, i
            if best_index < 0:
                break
            tokens[best_index:best_index + 2] = [tokens[best_index] + tokens[best_index + 1]]
        return [self.token_to_id.get(t, self.unk_id) for t in tokens]

    def encode_piece(self, piece):
        """Token ids of one piece of text WITHOUT the end-of-word token."""
        ids = self._cache.get(piece)
        if ids is None:
            ids = []
            for k in range(0, len(piece), 200):
                ids.extend(self._merge(piece[k:k + 200]))
            if len(self._cache) < 2_000_000:
                self._cache[piece] = ids
        return ids

def encode_sentence(text, tokenizer, max_tokens):
    """Token ids of one sentence exactly as the Phase-1 tokenizer encodes text: every whitespace-separated word is
    BPE-encoded and followed by the end-of-word token </w>. Sentences longer than max_tokens keep their beginning."""
    ids = []
    for word in str(text).split():
        ids.extend(tokenizer.encode_piece(word))
        ids.append(tokenizer.eow_id)
        if len(ids) >= max_tokens:
            break
    if not ids:
        ids = [tokenizer.eow_id]
    return ids[:max_tokens]

## 5. Metric

In [5]:
def rr_scores(gold, pred, keys=None):
    """gold / pred: {doc_key: [label ids]} -> macro precision / recall / F1, accuracy and per-role scores."""
    keys = list(gold) if keys is None else keys
    tp, n_pred, n_gold = Counter(), Counter(), Counter()
    correct = total = 0
    for key in keys:
        g, p = gold[key], pred[key]
        assert len(g) == len(p), (key, len(g), len(p))
        for a, b in zip(g, p):
            n_gold[a] += 1
            n_pred[b] += 1
            if a == b:
                tp[a] += 1
                correct += 1
            total += 1
    per_label = {}
    for k, lab in enumerate(LABELS):
        if n_gold[k] == 0 and n_pred[k] == 0:
            continue
        precision = tp[k] / n_pred[k] if n_pred[k] else 0.0
        recall = tp[k] / n_gold[k] if n_gold[k] else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        per_label[lab] = {"precision": precision, "recall": recall, "f1": f1, "gold": n_gold[k], "predicted": n_pred[k]}
    values = list(per_label.values())
    return {"macro_f1": float(np.mean([v["f1"] for v in values])) if values else 0.0,
            "macro_precision": float(np.mean([v["precision"] for v in values])) if values else 0.0,
            "macro_recall": float(np.mean([v["recall"] for v in values])) if values else 0.0,
            "accuracy": correct / max(1, total), "per_label": per_label}

def confusions(gold, pred, keys, top=10):
    c = Counter()
    for key in keys:
        for a, b in zip(gold[key], pred[key]):
            if a != b:
                c[(LABELS[a], LABELS[b])] += 1
    return [[a, b, n] for (a, b), n in c.most_common(top)]

_g = {1: [0, 1, 2, 2]}
assert rr_scores(_g, {1: [0, 1, 2, 2]})["macro_f1"] == 1.0
assert abs(rr_scores(_g, {1: [0, 0, 2, 2]})["macro_f1"] - np.mean([2 / 3, 0.0, 1.0])) < 1e-12
print("Scorer checked: macro-F1 over the roles present in gold or predictions.")

Scorer checked: macro-F1 over the roles present in gold or predictions.


## 6. Phase-1 models

In [6]:
ARCH_OF_TYPE = {"decoder-only": "A", "decoder-only-gqa": "B", "encoder-decoder": "C", "efficient-local-mqa": "D-alibi"}
ARCH_DESCRIPTION = {"A": "GPT decoder", "B": "GQA/RoPE decoder", "C": "encoder-decoder", "D-alibi": "local MQA/ALiBi decoder"}

def torch_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

def discover_phase1_checkpoints(root):
    """Finds the models written by the Phase-1 notebooks:
      <...>/<ARCH>/legal-model-final/{model.pt, config.json, tokenizer.json}       (normal case), or
      <...>/<ARCH>/best_model.pt + <...>/common/bpe_tokenizer_<ARCH>/tokenizer.json  (run stopped before the final save)."""
    found = []
    if not os.path.isdir(root):
        return found
    for dirpath, dirnames, filenames in os.walk(root):
        if dirpath[len(root):].count(os.sep) >= 12:
            dirnames[:] = []
        try:
            if os.path.basename(dirpath) == "legal-model-final" and "model.pt" in filenames and "tokenizer.json" in filenames:
                config = load_json_file(os.path.join(dirpath, "config.json"))
                history_path = os.path.join(dirpath, "training_history.json")
                model_file, tokenizer_dir = os.path.join(dirpath, "model.pt"), dirpath
                if os.path.exists(history_path):
                    eval_loss = float(load_json_file(history_path).get("best_eval_loss", "inf"))
                else:
                    eval_loss = float(torch_load(model_file).get("best_eval_loss", "inf"))
            elif "best_model.pt" in filenames and "legal-model-final" not in dirnames:
                common = os.path.join(os.path.dirname(dirpath), "common")
                tokenizer_dirs = [d for d in sorted(glob.glob(os.path.join(common, "bpe_tokenizer_" + os.path.basename(dirpath) + "*")))
                                  if os.path.exists(os.path.join(d, "tokenizer.json"))]
                if not tokenizer_dirs:
                    continue
                saved = torch_load(os.path.join(dirpath, "best_model.pt"))
                config, eval_loss = saved["config"], float(saved.get("eval_loss", "inf"))
                del saved
                model_file, tokenizer_dir = os.path.join(dirpath, "best_model.pt"), tokenizer_dirs[-1]
            else:
                continue
            arch = ARCH_OF_TYPE.get(config.get("type")) or str(config.get("architecture", "?"))[:1]
        except Exception as e:
            print("Skipping", dirpath, "(" + type(e).__name__ + ")")
            continue
        found.append({"arch": arch, "dir": dirpath, "model_file": model_file, "tokenizer_dir": tokenizer_dir, "config": config,
                      "eval_loss": eval_loss, "size_mb": os.path.getsize(model_file) / 1e6})
    return found

PHASE1_FOUND = discover_phase1_checkpoints(INPUT_ROOT)
PHASE1 = {}
print(f"Phase-1 checkpoints found under {INPUT_ROOT}: {len(PHASE1_FOUND)}")
for c in sorted(PHASE1_FOUND, key=lambda c: (c["arch"], c["eval_loss"])):
    print(f"  {ARCH_DESCRIPTION.get(c['arch'], c['arch']):24s} val loss {c['eval_loss']:.4f} | {c['size_mb']:.0f} MB | {c['dir']}")
_chosen = {}
for slot, spec in sorted(PHASE1_MODELS.items(), key=lambda kv: kv[1]["pick"] == "other"):
    arch, choice = spec["architecture"], spec["pick"]
    candidates = [c for c in PHASE1_FOUND if c["arch"] == arch]
    if choice == "other":
        taken = [_chosen[a]["eval_loss"] for a in _chosen if _chosen[a]["arch"] == arch]
        candidates = [c for c in candidates if all(abs(c["eval_loss"] - t) > 1e-6 for t in taken)]
        pick = max(candidates, key=lambda c: c["eval_loss"]) if candidates else None
    else:
        if choice not in ("best", "auto"):
            candidates = [c for c in candidates if choice.lower() in c["dir"].lower()]
        pick = min(candidates, key=lambda c: c["eval_loss"]) if candidates else None
    if pick is not None:
        PHASE1[slot] = _chosen[slot] = dict(pick, slot=slot)
print("Your Phase-1 models:")
for slot in PHASE1_MODELS:
    if slot in PHASE1:
        e = PHASE1[slot]
        print(f"  {slot}: {ARCH_DESCRIPTION.get(e['arch'], e['arch'])}, validation loss {e['eval_loss']:.4f} | {e['dir']}")
    else:
        print(f"  {slot}: NOT FOUND - attach the output of that Phase-1 notebook with 'Add Input'")

TOKENIZERS = {}

def tokenizer_for(slot):
    path = PHASE1[slot]["tokenizer_dir"]
    if path not in TOKENIZERS:
        TOKENIZERS[path] = BPETokenizer(path)
    return TOKENIZERS[path]

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        rms = torch.rsqrt(x.float().pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (x * rms.to(x.dtype)) * self.weight

class AttentionA(nn.Module):
    def __init__(self, d, n_heads, dropout):
        super().__init__()
        self.n_heads, self.head_dim, self.dropout = n_heads, d // n_heads, dropout
        self.qkv = nn.Linear(d, 3 * d, bias=True)
        self.out = nn.Linear(d, d, bias=True)

    def forward(self, x):
        b, t, c = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q, k, v = [z.view(b, t, self.n_heads, self.head_dim).transpose(1, 2) for z in (q, k, v)]
        y = F.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        return self.out(y.transpose(1, 2).contiguous().view(b, t, c))

class BlockA(nn.Module):
    def __init__(self, d, n_heads, d_ff, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.attn = AttentionA(d, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d)
        self.fc1 = nn.Linear(d, d_ff)
        self.fc2 = nn.Linear(d_ff, d)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.dropout(self.attn(self.ln1(x)))
        return x + self.dropout(self.fc2(F.gelu(self.fc1(self.ln2(x)))))

class BackboneA(nn.Module):
    bidirectional = False

    def __init__(self, cfg, vocab_size, n_layers, dropout):
        super().__init__()
        d = cfg["d_model"]
        self.d_model, self.max_tokens = d, cfg["context_length"]
        self.token_embedding = nn.Embedding(vocab_size, d)
        self.position_embedding = nn.Embedding(cfg["context_length"], d)
        self.blocks = nn.ModuleList([BlockA(d, cfg["n_head"], cfg["d_ff"], dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d)

    def layers(self):
        return self.blocks

    def forward(self, input_ids, pad_mask=None):
        positions = torch.arange(input_ids.size(1), device=input_ids.device)
        x = self.token_embedding(input_ids) + self.position_embedding(positions)[None]
        states = []
        for block in self.blocks:
            x = block(x)
            states.append(self.ln_f(x))
        return states

def rotate_half(x):
    return torch.stack((-x[..., 1::2], x[..., ::2]), dim=-1).flatten(-2)

class AttentionB(nn.Module):
    def __init__(self, d, n_heads, n_kv_heads, max_len, dropout):
        super().__init__()
        self.q_heads, self.kv_heads, self.head_dim, self.dropout = n_heads, n_kv_heads, d // n_heads, dropout
        self.q_proj = nn.Linear(d, d, bias=False)
        self.k_proj = nn.Linear(d, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d, n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(d, d, bias=False)
        inv_freq = 1.0 / (10000.0 ** (torch.arange(0, self.head_dim, 2).float() / self.head_dim))
        freqs = torch.outer(torch.arange(max_len).float(), inv_freq)
        self.register_buffer("cos", torch.cos(freqs).repeat_interleave(2, dim=-1), persistent=False)
        self.register_buffer("sin", torch.sin(freqs).repeat_interleave(2, dim=-1), persistent=False)

    def forward(self, x):
        b, t, c = x.shape
        q = self.q_proj(x).view(b, t, self.q_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(b, t, self.kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(b, t, self.kv_heads, self.head_dim).transpose(1, 2)
        cos, sin = self.cos[:t].to(q.dtype)[None, None], self.sin[:t].to(q.dtype)[None, None]
        q = q * cos + rotate_half(q) * sin
        k = k * cos + rotate_half(k) * sin
        repeat = self.q_heads // self.kv_heads
        if repeat > 1:
            k, v = k.repeat_interleave(repeat, dim=1), v.repeat_interleave(repeat, dim=1)
        y = F.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        return self.o_proj(y.transpose(1, 2).contiguous().view(b, t, c))

class SwiGLU(nn.Module):
    def __init__(self, d, d_ff):
        super().__init__()
        self.gate = nn.Linear(d, d_ff, bias=False)
        self.up = nn.Linear(d, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

class BlockB(nn.Module):
    def __init__(self, d, n_heads, n_kv_heads, d_ff, max_len, dropout):
        super().__init__()
        self.norm1 = RMSNorm(d)
        self.attn = AttentionB(d, n_heads, n_kv_heads, max_len, dropout)
        self.norm2 = RMSNorm(d)
        self.mlp = SwiGLU(d, d_ff)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.dropout(self.attn(self.norm1(x)))
        return x + self.dropout(self.mlp(self.norm2(x)))

class BackboneB(nn.Module):
    bidirectional = False

    def __init__(self, cfg, vocab_size, n_layers, dropout):
        super().__init__()
        d = cfg["d_model"]
        self.d_model, self.max_tokens = d, cfg["context_length"]
        self.token_embedding = nn.Embedding(vocab_size, d)
        self.blocks = nn.ModuleList([BlockB(d, cfg["n_head"], cfg["n_kv_head"], cfg["d_ff"], cfg["context_length"], dropout)
                                     for _ in range(n_layers)])
        self.norm_f = RMSNorm(d)

    def layers(self):
        return self.blocks

    def forward(self, input_ids, pad_mask=None):
        x = self.token_embedding(input_ids)
        states = []
        for block in self.blocks:
            x = block(x)
            states.append(self.norm_f(x))
        return states

class AttentionC(nn.Module):
    def __init__(self, d, n_heads, dropout):
        super().__init__()
        self.n_heads, self.head_dim, self.dropout = n_heads, d // n_heads, dropout
        self.q_proj = nn.Linear(d, d)
        self.k_proj = nn.Linear(d, d)
        self.v_proj = nn.Linear(d, d)
        self.o_proj = nn.Linear(d, d)

    def forward(self, x, attn_mask=None):
        b, t, c = x.shape
        q, k, v = [proj(x).view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
                   for proj in (self.q_proj, self.k_proj, self.v_proj)]
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, dropout_p=self.dropout if self.training else 0.0)
        return self.o_proj(y.transpose(1, 2).contiguous().view(b, t, c))

class EncoderBlockC(nn.Module):
    def __init__(self, d, n_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = AttentionC(d, n_heads, dropout)
        self.ln1 = nn.LayerNorm(d)
        self.ff1 = nn.Linear(d, d_ff)
        self.ff2 = nn.Linear(d_ff, d)
        self.ln2 = nn.LayerNorm(d)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attn_mask=None):
        x = self.ln1(x + self.dropout(self.self_attn(x, attn_mask)))
        return self.ln2(x + self.dropout(self.ff2(F.gelu(self.ff1(x)))))

class BackboneC(nn.Module):
    bidirectional = True

    def __init__(self, cfg, vocab_size, n_layers, dropout):
        super().__init__()
        d = cfg["d_model"]
        self.d_model, self.max_tokens = d, cfg["encoder_length"]
        self.token_embedding = nn.Embedding(vocab_size, d)
        self.encoder = nn.ModuleList([EncoderBlockC(d, cfg["n_head"], cfg["d_ff"], dropout) for _ in range(n_layers)])
        pe = torch.zeros(self.max_tokens, d)
        position = torch.arange(0, self.max_tokens).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe, persistent=False)

    def layers(self):
        return self.encoder

    embedding_state = False

    def forward(self, input_ids, pad_mask=None):
        x = self.token_embedding(input_ids)
        states = []
        if self.embedding_state:
            scale = self.token_embedding.weight.detach().float().pow(2).mean().sqrt().clamp_min(1e-12)
            states.append(F.layer_norm(x / scale.to(x.dtype), (self.d_model,)))
        x = x + self.pe[:x.size(1)].to(x.dtype)[None]
        attn_mask = None if pad_mask is None else pad_mask[:, None, None, :]
        for block in self.encoder:
            x = block(x, attn_mask)
            states.append(x)
        return states

class AttentionD(nn.Module):
    def __init__(self, d, n_heads, dropout):
        super().__init__()
        self.n_heads, self.head_dim, self.dropout = n_heads, d // n_heads, dropout
        self.q_proj = nn.Linear(d, d, bias=False)
        self.k_proj = nn.Linear(d, self.head_dim, bias=False)
        self.v_proj = nn.Linear(d, self.head_dim, bias=False)
        self.o_proj = nn.Linear(d, d, bias=False)

    def forward(self, x, attn_bias):
        b, t, c = x.shape
        q = self.q_proj(x).view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(b, t, 1, self.head_dim).transpose(1, 2).expand(-1, self.n_heads, -1, -1)
        v = self.v_proj(x).view(b, t, 1, self.head_dim).transpose(1, 2).expand(-1, self.n_heads, -1, -1)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_bias.to(q.dtype),
                                           dropout_p=self.dropout if self.training else 0.0)
        return self.o_proj(y.transpose(1, 2).contiguous().view(b, t, c))

class BlockD(nn.Module):
    def __init__(self, d, n_heads, d_ff, dropout):
        super().__init__()
        self.norm1 = RMSNorm(d)
        self.attn = AttentionD(d, n_heads, dropout)
        self.norm2 = RMSNorm(d)
        self.fc1 = nn.Linear(d, d_ff)
        self.fc2 = nn.Linear(d_ff, d)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attn_bias):
        x = x + self.dropout(self.attn(self.norm1(x), attn_bias))
        return x + self.dropout(self.fc2(F.gelu(self.fc1(self.norm2(x)))))

class BackboneD(nn.Module):
    bidirectional = False

    def __init__(self, cfg, vocab_size, n_layers, dropout):
        super().__init__()
        d = cfg["d_model"]
        self.d_model, self.max_tokens, self.window = d, cfg["context_length"], cfg["local_window"]
        self.token_embedding = nn.Embedding(vocab_size, d)
        self.blocks = nn.ModuleList([BlockD(d, cfg["n_head"], cfg["d_ff"], dropout) for _ in range(n_layers)])
        self.norm_f = RMSNorm(d)
        n = cfg["n_head"]
        self.register_buffer("slopes", 2.0 ** (-(8.0 * torch.arange(n) / n)), persistent=False)

    def layers(self):
        return self.blocks

    def attention_bias(self, t, device):
        pos = torch.arange(t, device=device, dtype=torch.float32)
        distance = pos[:, None] - pos[None, :]
        blocked = (distance < 0) | (distance > self.window - 1)
        bias = -self.slopes.to(device)[:, None, None] * distance[None]
        return bias.masked_fill(blocked[None], float("-inf"))[None]

    def forward(self, input_ids, pad_mask=None):
        x = self.token_embedding(input_ids)
        attn_bias = self.attention_bias(input_ids.size(1), input_ids.device)
        states = []
        for block in self.blocks:
            x = block(x, attn_bias)
            states.append(self.norm_f(x))
        return states

BACKBONE_CLASSES = {"A": BackboneA, "B": BackboneB, "C": BackboneC, "D-alibi": BackboneD}

@torch.no_grad()
def encoder_token_check(backbone, length=256):
    """Does the Phase-1 encoder output depend on the input tokens? Two different random token sequences are encoded;
    if the last-layer states at the same positions are almost identical, the encoder only encodes positions."""
    was_training = backbone.training
    backbone.eval()
    g = torch.Generator().manual_seed(0)
    vocab = backbone.token_embedding.weight.shape[0]
    length = min(length, backbone.max_tokens)
    ids = torch.randint(4, vocab, (2, length), generator=g)
    flag, backbone.embedding_state = backbone.embedding_state, False
    last = backbone(ids)[-1].float()
    backbone.embedding_state = flag
    same_position = F.cosine_similarity(last[0], last[1], dim=-1).mean().item()
    other_position = F.cosine_similarity(last[0], last[0].roll(1, 0), dim=-1).mean().item()
    emb_rms = backbone.token_embedding.weight.float().pow(2).mean().sqrt().item()
    pe_rms = backbone.pe[:length].float().pow(2).mean().sqrt().item()
    print(f"Encoder check: token-embedding RMS {emb_rms:.4f} vs position-encoding RMS {pe_rms:.4f} | last layer, "
          f"cosine similarity of two DIFFERENT inputs at the same position {same_position:.3f} "
          f"(neighbouring positions of one input {other_position:.3f}; close to 1.0 = the output ignores the tokens)")
    backbone.train(was_training)

def load_backbone(slot, n_layers=None, dropout=0.1, verbose=True):
    """Build the architecture and load its Phase-1 weights (optionally only the first n layers)."""
    entry = PHASE1[slot]
    arch = entry["arch"]
    cfg = entry["config"]
    state = torch_load(entry["model_file"])["model"]
    total_layers = cfg["encoder_layers"] if arch == "C" else cfg["n_layer"]
    n_layers = total_layers if n_layers is None else min(int(n_layers), total_layers)
    vocab_size = state["token_embedding.weight"].shape[0]
    backbone = BACKBONE_CLASSES[arch](cfg, vocab_size, n_layers, dropout)
    wanted = set(backbone.state_dict().keys())
    missing = wanted - set(state.keys())
    if missing:
        raise RuntimeError(f"Phase-1 checkpoint for {slot} lacks weights: {sorted(missing)[:5]} ...")
    backbone.load_state_dict({k: v for k, v in state.items() if k in wanted}, strict=True)
    if verbose and arch == "C":
        encoder_token_check(backbone)
    if verbose:
        unused = len(state) - len(wanted)
        print(f"Loaded Phase-1 model {slot} ({ARCH_DESCRIPTION.get(arch, arch)}): {n_layers}/{total_layers} layers, "
              f"{sum(p.numel() for p in backbone.parameters()) / 1e6:.2f}M parameters"
              + (f" ({unused} weight tensors not needed, e.g. removed layers or the decoder)" if unused else ""))
    return backbone

Phase-1 checkpoints found under /kaggle/input: 4
  GPT decoder              val loss 1.3086 | 258 MB | /kaggle/input/datasets/prathamk26/cs781-phase1/untitled folder/results-D/cs781_phase1/A_GPT_MHA_LearnedPos/legal-model-final
  GPT decoder              val loss 1.7840 | 258 MB | /kaggle/input/datasets/prathamk26/cs781-phase1/untitled folder/results-A/cs781_phase1/A_GPT_MHA_LearnedPos/legal-model-final
  GQA/RoPE decoder         val loss 1.6521 | 264 MB | /kaggle/input/datasets/prathamk26/cs781-phase1/untitled folder/results-B/cs781_phase1/B_GQA_RoPE_RMSNorm_SwiGLU/legal-model-final
  encoder-decoder          val loss 3.0858 | 422 MB | /kaggle/input/datasets/prathamk26/cs781-phase1/untitled folder/results-C/cs781_phase1/C_EncoderDecoder/legal-model-final
Your Phase-1 models:
  A: GPT decoder, validation loss 1.7840 | /kaggle/input/datasets/prathamk26/cs781-phase1/untitled folder/results-A/cs781_phase1/A_GPT_MHA_LearnedPos/legal-model-final
  B: GQA/RoPE decoder, validation loss 1.6521

## 7. Fine-tuning and adaptation techniques

| Technique | What is trained |
|---|---|
| `full` | every weight |
| `lora` | low-rank update added to q/v: W0 + (α/r)·BA (`lora_scaling="rs"`: α/√r, rsLoRA) |
| `lorma` | low-rank multiplicative update: (I + (α/r)·BA)·W0 (LoRMA+), or P(BA)·W0 (LoRMA-π) |
| `ia3` | one learned scale per key, value and feed-forward unit (IA³) |
| `adapter` / `bitfit` / `frozen` | bottleneck adapters / biases + norms / head only |

In [7]:
class LoRALinear(nn.Module):
    """LoRA (Hu et al.): additive low-rank update,  W = W0 + (alpha / r) * B A.   B starts at zero.
    With scaling="rs" it is rsLoRA:  W = W0 + (alpha / sqrt(r)) * B A."""

    def __init__(self, base, r, alpha, rows=None, scaling="standard"):
        super().__init__()
        self.base, self.scale = base, (alpha / math.sqrt(r) if scaling == "rs" else alpha / r)
        self.rows = rows or [(0, base.out_features)]
        self.lora_A = nn.ParameterList([nn.Parameter(torch.randn(r, base.in_features) / math.sqrt(base.in_features))
                                        for _ in self.rows])
        self.lora_B = nn.ParameterList([nn.Parameter(torch.zeros(e - s, r)) for s, e in self.rows])

    def forward(self, x):
        y = self.base(x)
        parts, cursor = [], 0
        for (s, e), A, B in zip(self.rows, self.lora_A, self.lora_B):
            if s > cursor:
                parts.append(y[..., cursor:s])
            parts.append(y[..., s:e] + self.scale * F.linear(F.linear(x, A.to(x.dtype)), B.to(x.dtype)))
            cursor = e
        if cursor < y.size(-1):
            parts.append(y[..., cursor:])
        return parts[0] if len(parts) == 1 else torch.cat(parts, dim=-1)

_PERM_INDEX = {}

class LoRMALinear(nn.Module):
    """LoRMA (Low-Rank Multiplicative Adaptation): the pretrained weight is MULTIPLIED by a learned matrix.

      "plus" (LoRMA+):   W = (I + (alpha / r) * B A) W0     B starts at zero, so the multiplier starts as the identity.
      "pi"   (LoRMA-pi): W = P(B A) W0, where P shifts row i of (B A) cyclically by i positions, which makes a
                         low-rank product full rank. B's first column is ones and A[0, 0] = 1 (rest of A zero), so
                         P(B A) starts as the identity.
    Because the multiplier is on the left of W0, it acts on the layer's OUTPUT: h = W0 x, y = M h + bias.
    """

    def __init__(self, base, r, alpha, rows=None, variant="plus"):
        super().__init__()
        self.base, self.scale, self.variant = base, alpha / r, variant
        self.rows = rows or [(0, base.out_features)]
        self.lorma_A, self.lorma_B = nn.ParameterList(), nn.ParameterList()
        for s, e in self.rows:
            m = e - s
            if variant == "plus":
                A, B = torch.randn(r, m) / math.sqrt(m), torch.zeros(m, r)
            elif variant == "pi":
                A, B = torch.zeros(r, m), torch.randn(m, r) * 0.02
                A[0, 0] = 1.0
                B[:, 0] = 1.0
            else:
                raise ValueError("lorma_variant must be 'plus' or 'pi'")
            self.lorma_A.append(nn.Parameter(A))
            self.lorma_B.append(nn.Parameter(B))

    @staticmethod
    def _permuted(BA):
        m = BA.size(0)
        key = (m, BA.device)
        if key not in _PERM_INDEX:
            i = torch.arange(m, device=BA.device)
            _PERM_INDEX[key] = (i[None, :] - i[:, None]) % m
        return torch.gather(BA, 1, _PERM_INDEX[key])

    def forward(self, x):
        h = F.linear(x, self.base.weight)
        parts, cursor = [], 0
        for (s, e), A, B in zip(self.rows, self.lorma_A, self.lorma_B):
            if s > cursor:
                parts.append(h[..., cursor:s])
            hs = h[..., s:e]
            if self.variant == "plus":
                parts.append(hs + self.scale * F.linear(F.linear(hs, A.to(hs.dtype)), B.to(hs.dtype)))
            else:
                parts.append(F.linear(hs, self._permuted(B @ A).to(hs.dtype)))
            cursor = e
        if cursor < h.size(-1):
            parts.append(h[..., cursor:])
        y = parts[0] if len(parts) == 1 else torch.cat(parts, dim=-1)
        return y if self.base.bias is None else y + self.base.bias

class AdapterLinear(nn.Module):
    """Houlsby-style bottleneck adapter placed on a sub-layer's output, before the residual connection:
    y = h + up(GELU(down(h))),  h = base(x).   `up` starts at zero, so the adapter starts as the identity."""

    def __init__(self, base, bottleneck):
        super().__init__()
        self.base = base
        self.adapter_down = nn.Linear(base.out_features, bottleneck)
        self.adapter_up = nn.Linear(bottleneck, base.out_features)
        nn.init.normal_(self.adapter_down.weight, std=0.02)
        nn.init.zeros_(self.adapter_down.bias)
        nn.init.zeros_(self.adapter_up.weight)
        nn.init.zeros_(self.adapter_up.bias)

    def forward(self, x):
        h = self.base(x)
        return h + self.adapter_up(F.gelu(self.adapter_down(h)))

class IA3Linear(nn.Module):
    """IA3 (Liu et al., "(IA)^3"): learned vectors that RESCALE activations, initialised to ones (= no change).
    mode "out": rescale the layer's output (used for attention keys and values; `rows` limits it to part of a fused
    matrix).  mode "in": rescale the layer's input (used for the feed-forward hidden activations, before fc2)."""

    def __init__(self, base, mode, rows=None):
        super().__init__()
        self.base, self.mode = base, mode
        if mode == "in":
            self.ia3_scale = nn.Parameter(torch.ones(base.in_features))
            self.rows = None
        else:
            self.rows = rows or [(0, base.out_features)]
            index = torch.cat([torch.arange(s, e) for s, e in self.rows])
            self.register_buffer("row_index", index, persistent=False)
            self.ia3_scale = nn.Parameter(torch.ones(len(index)))

    def forward(self, x):
        if self.mode == "in":
            return self.base(x * self.ia3_scale.to(x.dtype))
        y = self.base(x)
        full = torch.ones(y.size(-1), device=y.device, dtype=y.dtype).scatter(0, self.row_index, self.ia3_scale.to(y.dtype))
        return y * full

QV_TARGETS = {"A": ["attn.qkv"], "B": ["attn.q_proj", "attn.v_proj"], "C": ["self_attn.q_proj", "self_attn.v_proj"],
              "D-alibi": ["attn.q_proj", "attn.v_proj"]}
ALL_TARGETS = {"A": ["attn.qkv", "attn.out", "fc1", "fc2"],
               "B": ["attn.q_proj", "attn.k_proj", "attn.v_proj", "attn.o_proj", "mlp.gate", "mlp.up", "mlp.down"],
               "C": ["self_attn.q_proj", "self_attn.k_proj", "self_attn.v_proj", "self_attn.o_proj", "ff1", "ff2"],
               "D-alibi": ["attn.q_proj", "attn.k_proj", "attn.v_proj", "attn.o_proj", "fc1", "fc2"]}
SUBLAYER_OUTPUTS = {"A": ["attn.out", "fc2"], "B": ["attn.o_proj", "mlp.down"], "C": ["self_attn.o_proj", "ff2"],
                    "D-alibi": ["attn.o_proj", "fc2"]}
IA3_TARGETS = {"A": [("attn.qkv", "out"), ("fc2", "in")], "B": [("attn.k_proj", "out"), ("attn.v_proj", "out"), ("mlp.down", "in")],
               "C": [("self_attn.k_proj", "out"), ("self_attn.v_proj", "out"), ("ff2", "in")],
               "D-alibi": [("attn.k_proj", "out"), ("attn.v_proj", "out"), ("fc2", "in")]}
NEW_WEIGHT_MARKERS = ("lora_", "lorma_", "adapter_", "ia3_")

def _get(module, path):
    for part in path.split("."):
        module = getattr(module, part)
    return module

def _set(module, path, value):
    parts = path.split(".")
    for part in parts[:-1]:
        module = getattr(module, part)
    setattr(module, parts[-1], value)

def apply_technique(backbone, arch, cfg):
    """Freeze the pretrained model, then make trainable exactly what the chosen technique allows."""
    technique = cfg["technique"]
    for p in backbone.parameters():
        p.requires_grad = False
    blocks = backbone.layers()
    norm_types = (nn.LayerNorm, RMSNorm)
    if technique == "full":
        for p in backbone.parameters():
            p.requires_grad = True
    elif technique == "partial":
        for block in list(blocks)[-int(cfg["partial_top_layers"]):]:
            for p in block.parameters():
                p.requires_grad = True
        for name in ("ln_f", "norm_f"):
            if hasattr(backbone, name):
                for p in getattr(backbone, name).parameters():
                    p.requires_grad = True
    elif technique == "bitfit":
        for name, p in backbone.named_parameters():
            if name.endswith(".bias"):
                p.requires_grad = True
        for module in backbone.modules():
            if isinstance(module, norm_types):
                for p in module.parameters():
                    p.requires_grad = True
    elif technique in ("lora", "lorma"):
        targets = (QV_TARGETS if cfg["peft_targets"] == "qv" else ALL_TARGETS)[arch]
        for block in blocks:
            for path in targets:
                base = _get(block, path)
                rows = None
                if arch == "A" and path == "attn.qkv" and cfg["peft_targets"] == "qv":
                    d = base.in_features
                    rows = [(0, d), (2 * d, 3 * d)]
                if technique == "lora":
                    wrapped = LoRALinear(base, cfg["lora_r"], cfg["lora_alpha"], rows, cfg.get("lora_scaling", "standard"))
                else:
                    wrapped = LoRMALinear(base, cfg["lora_r"], cfg["lora_alpha"], rows, cfg["lorma_variant"])
                _set(block, path, wrapped)
    elif technique == "ia3":
        for block in blocks:
            for path, mode in IA3_TARGETS[arch]:
                base = _get(block, path)
                rows = None
                if arch == "A" and path == "attn.qkv":
                    d = base.in_features
                    rows = [(d, 2 * d), (2 * d, 3 * d)]
                _set(block, path, IA3Linear(base, mode, rows))
    elif technique == "adapter":
        for block in blocks:
            for path in SUBLAYER_OUTPUTS[arch]:
                _set(block, path, AdapterLinear(_get(block, path), cfg["adapter_dim"]))
        for module in backbone.modules():
            if isinstance(module, norm_types):
                for p in module.parameters():
                    p.requires_grad = True
    elif technique != "frozen":
        raise ValueError("Unknown technique: " + str(technique))
    for name, p in backbone.named_parameters():
        if any(marker in name for marker in NEW_WEIGHT_MARKERS):
            p.requires_grad = True
    return backbone

## 8. Model: sentence vectors → document BiLSTM → CRF

In [8]:
class RoleCRF(nn.Module):
    """Linear-chain CRF: learns how likely each role is to follow another (e.g. Fact -> Fact, Ratio -> Ruling).
    Training uses chunks that can start in the middle of a document, so there are no start/end scores."""

    def __init__(self, n):
        super().__init__()
        self.transitions = nn.Parameter(torch.zeros(n, n))

    def nll(self, emissions, tags, mask):
        """Negative log-likelihood summed over the batch. emissions [B, S, K], tags [B, S], mask [B, S] (bool)."""
        batch = torch.arange(emissions.size(0), device=emissions.device)
        fmask = mask.float()
        score = emissions[batch, 0, tags[:, 0]]
        alpha = emissions[:, 0]
        for t in range(1, emissions.size(1)):
            score = score + (emissions[batch, t, tags[:, t]] + self.transitions[tags[:, t - 1], tags[:, t]]) * fmask[:, t]
            nxt = torch.logsumexp(alpha[:, :, None] + self.transitions[None], dim=1) + emissions[:, t]
            alpha = torch.where(mask[:, t, None], nxt, alpha)
        return (torch.logsumexp(alpha, dim=1) - score).sum()

    def decode(self, emissions):
        """Best role sequence for one document. emissions: numpy [S, K]."""
        trans = self.transitions.detach().float().cpu().numpy().astype(np.float64)
        em = emissions.astype(np.float64)
        n, k = em.shape
        score = em[0].copy()
        back = np.zeros((n, k), dtype=np.int64)
        for t in range(1, n):
            cand = score[:, None] + trans
            back[t] = cand.argmax(0)
            score = cand.max(0) + em[t]
        path = [int(score.argmax())]
        for t in range(n - 1, 0, -1):
            path.append(int(back[t, path[-1]]))
        return path[::-1]

class RRModel(nn.Module):
    """Phase-1 language model + rhetorical-role head.

    1. Sentence vector: the language model reads one sentence; a learned weighted average of ALL its layers
       ("scalar mix") is averaged over the sentence's tokens.
    2. Domain: a learned CL or IT vector is added (one model serves both domains).
    3. Document context: a BiLSTM runs over the sequence of sentence vectors, so each sentence sees what comes before
       and after it (context="none" skips this: every sentence is classified on its own).
    4. A linear layer scores the 13 roles; a CRF picks the best role SEQUENCE (or softmax: the best role per sentence).
    """

    def __init__(self, backbone, cfg):
        super().__init__()
        self.backbone, self.decoder_type = backbone, cfg["decoder"]
        d = backbone.d_model
        self.mix = nn.Parameter(torch.zeros(len(backbone.layers()) + int(getattr(backbone, "embedding_state", False))))
        self.domain = nn.Embedding(len(DOMAINS), d) if cfg["domain_embedding"] else None
        if self.domain is not None:
            nn.init.zeros_(self.domain.weight)
        self.norm = nn.LayerNorm(d)
        self.drop = nn.Dropout(cfg["head_dropout"])
        self.lstm = None
        if cfg["context"] == "bilstm":
            self.lstm = nn.LSTM(d, cfg["lstm_hidden"], num_layers=cfg["lstm_layers"], batch_first=True, bidirectional=True)
            d = 2 * cfg["lstm_hidden"]
        self.out = nn.Linear(d, NUM_LABELS)
        self.crf = RoleCRF(NUM_LABELS) if self.decoder_type == "crf" else None

    def sentence_vectors(self, input_ids, pad_mask):
        """[N, T] token ids of N sentences -> [N, d] float32 sentence vectors."""
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            states = self.backbone(input_ids, pad_mask)
        weights = torch.softmax(self.mix, dim=0)
        assert len(states) == len(weights), (len(states), len(weights))
        h = sum(w * s.float() for w, s in zip(weights, states))
        m = pad_mask[:, :, None].float()
        return (h * m).sum(1) / m.sum(1).clamp(min=1.0)

    def head(self, vectors, lengths, domains):
        """vectors [B, S, d] (sentences of B document chunks), lengths [B], domains [B] -> role scores [B, S, K]."""
        x = vectors
        if self.domain is not None:
            x = x + self.domain(domains)[:, None, :]
        x = self.drop(self.norm(x))
        if self.lstm is not None:
            packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
            x, _ = nn.utils.rnn.pad_packed_sequence(self.lstm(packed)[0], batch_first=True, total_length=vectors.size(1))
            x = self.drop(x)
        return self.out(x)

    def loss(self, scores, tags, mask, cfg, class_weights=None, teacher=None):
        n = mask.sum().clamp(min=1)
        if self.decoder_type == "crf":
            hard = self.crf.nll(scores, tags, mask) / n
        else:
            hard = F.cross_entropy(scores[mask], tags[mask], weight=class_weights)
        if teacher is None:
            return hard, hard.detach(), None
        T = float(cfg["kd_temperature"])
        soft = F.kl_div(F.log_softmax(scores[mask] / T, dim=-1), F.softmax(teacher[mask] / T, dim=-1),
                        reduction="batchmean") * T * T
        w = float(cfg["kd_weight"])
        return (1 - w) * hard + w * soft, hard.detach(), soft.detach()

    def decode(self, scores):
        """Role scores of one whole document [S, K] (numpy) -> list of role ids."""
        if len(scores) == 0:
            return []
        if self.crf is not None:
            return self.crf.decode(scores)
        return [int(i) for i in scores.argmax(1)]

## 9. Training and prediction

Training uses chunks of consecutive sentences; prediction labels each whole document in one pass.

In [9]:
import gc

ENCODINGS = {}

def encodings_for(slot, keys, max_tokens):
    """{doc_key: [np.int64 token ids of each sentence]} for the tokenizer of Phase-1 model `slot`."""
    tokenizer = tokenizer_for(slot)
    cache = ENCODINGS.setdefault((PHASE1[slot]["tokenizer_dir"], max_tokens), {})
    missing = [k for k in keys if k not in cache]
    for key in tqdm(missing, desc=f"tokenizing for model {slot}", disable=len(missing) < 5):
        cache[key] = [np.asarray(encode_sentence(s, tokenizer, max_tokens), dtype=np.int64) for s in DOCS[key]["text"]]
    return cache

def training_chunks(keys, size, seed):
    """Cut every training document into runs of `size` consecutive sentences, starting at a random offset so the
    chunk borders move from epoch to epoch. -> list of (doc_key, first, last + 1)."""
    rng = random.Random(seed)
    out = []
    for key in keys:
        n = len(DOCS[key]["text"])
        first = rng.randrange(size) if n > size else 0
        starts = ([0] if first > 0 else []) + list(range(first, n, size))
        for i, s in enumerate(starts):
            e = starts[i + 1] if i + 1 < len(starts) else n
            out.append((key, s, e))
    return out

def token_batch(sentences):
    """List of token-id arrays -> padded ids [N, T] and mask [N, T] on the device."""
    width = max(len(s) for s in sentences)
    ids = np.full((len(sentences), width), PAD_ID, dtype=np.int64)
    mask = np.zeros((len(sentences), width), dtype=bool)
    for i, s in enumerate(sentences):
        ids[i, :len(s)], mask[i, :len(s)] = s, True
    return torch.from_numpy(ids).to(DEVICE), torch.from_numpy(mask).to(DEVICE)

def chunk_forward(model, chunks, enc, teacher=None):
    """Forward pass of a training batch of chunks -> scores [B, S, K], tags, mask (and teacher scores)."""
    sentences, rows, cols = [], [], []
    for b, (key, s, e) in enumerate(chunks):
        for j in range(s, e):
            sentences.append(enc[key][j])
            rows.append(b)
            cols.append(j - s)
    ids, tmask = token_batch(sentences)
    vec = model.sentence_vectors(ids, tmask)
    width = max(e - s for _, s, e in chunks)
    vectors = vec.new_zeros(len(chunks), width, vec.size(-1))
    vectors[torch.tensor(rows, device=DEVICE), torch.tensor(cols, device=DEVICE)] = vec
    lengths = torch.tensor([e - s for _, s, e in chunks])
    domains = torch.tensor([DOMAINS.index(DOCS[k]["domain"]) for k, _, _ in chunks], device=DEVICE)
    scores = model.head(vectors, lengths, domains)
    tags = torch.zeros(len(chunks), width, dtype=torch.long)
    mask = torch.zeros(len(chunks), width, dtype=torch.bool)
    t = torch.zeros(len(chunks), width, NUM_LABELS) if teacher is not None else None
    for b, (key, s, e) in enumerate(chunks):
        tags[b, :e - s] = torch.tensor(DOCS[key]["labels"][s:e])
        mask[b, :e - s] = True
        if t is not None:
            t[b, :e - s] = torch.from_numpy(teacher[key][s:e])
    return scores, tags.to(DEVICE), mask.to(DEVICE), (t.to(DEVICE) if t is not None else None)

@torch.no_grad()
def predict_scores(model, cfg, keys, enc, desc="predict"):
    """Role scores for whole documents: all sentences are encoded (in batches, sorted by length), then each document
    goes through the head in ONE pass, so every sentence sees its full document. -> {doc_key: float32 [S, K]}."""
    model.eval()
    items = sorted(((len(enc[k][j]), k, j) for k in keys for j in range(len(enc[k]))), key=lambda x: x[0])
    vectors = {k: torch.zeros(len(enc[k]), model.backbone.d_model) for k in keys}
    size = cfg["eval_sentence_batch"]
    for i in tqdm(range(0, len(items), size), desc=desc, leave=False):
        chunk = items[i:i + size]
        ids, tmask = token_batch([enc[k][j] for _, k, j in chunk])
        vec = model.sentence_vectors(ids, tmask).cpu()
        for (_, k, j), v in zip(chunk, vec):
            vectors[k][j] = v
    out = {}
    for k in keys:
        v = vectors[k][None].to(DEVICE)
        domains = torch.tensor([DOMAINS.index(DOCS[k]["domain"])], device=DEVICE)
        out[k] = model.head(v, torch.tensor([v.size(1)]), domains)[0].float().cpu().numpy()
    return out

def decode_all(model, scores):
    return {k: model.decode(s) for k, s in scores.items()}

def evaluate(model, cfg, keys, enc, desc="evaluate"):
    pred = decode_all(model, predict_scores(model, cfg, keys, enc, desc))
    gold = {k: DOCS[k]["labels"] for k in keys}
    metrics = rr_scores(gold, pred, keys)
    for domain in DOMAINS:
        dk = [k for k in keys if k[1] == domain]
        if dk:
            metrics["macro_f1_" + domain] = rr_scores(gold, pred, dk)["macro_f1"]
    return metrics, pred

def exp_dir(name):
    path = os.path.join(WORK_ROOT, name)
    os.makedirs(path, exist_ok=True)
    return path

def build_model(cfg, verbose=True):
    backbone = load_backbone(cfg["backbone"], cfg["n_layers"], cfg["dropout"], verbose=verbose)
    apply_technique(backbone, PHASE1[cfg["backbone"]]["arch"], cfg)
    if cfg.get("embedding_state"):
        if not hasattr(backbone, "embedding_state"):
            raise ValueError("embedding_state is only implemented for the encoder (model C)")
        backbone.embedding_state = True
    return RRModel(backbone, cfg).to(DEVICE)

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    backbone = sum(p.numel() for p in model.backbone.parameters())
    backbone_trainable = sum(p.numel() for p in model.backbone.parameters() if p.requires_grad)
    return {"total": total, "trainable": trainable, "trainable_percent": 100.0 * trainable / total,
            "language_model": backbone, "language_model_trainable": backbone_trainable, "head": total - backbone}

def trainable_state(model):
    return {name: p.detach().cpu().clone() for name, p in model.named_parameters() if p.requires_grad}

def load_trainable(model, state):
    result = model.load_state_dict(state, strict=False)
    if result.unexpected_keys:
        raise RuntimeError("Saved weights do not fit this model: " + str(result.unexpected_keys[:5]))

def save_atomic(obj, path):
    torch.save(obj, path + ".tmp")
    os.replace(path + ".tmp", path)

def config_signature(cfg, extra=""):
    entry = PHASE1[cfg["backbone"]]
    payload = {"cfg": {k: v for k, v in cfg.items() if k != "name"}, "phase1": [entry["config"], entry["eval_loss"]],
               "labels": LABELS, "extra": extra}
    return hashlib.md5(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()

def optimizer_groups(model, cfg):
    backbone_lr = cfg["lr_peft"] if cfg["technique"] in ("bitfit", "adapter", "ia3") else cfg["lr_backbone"]
    buckets = defaultdict(list)
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if not name.startswith("backbone."):
            lr = cfg["lr_head"]
        elif any(marker in name for marker in NEW_WEIGHT_MARKERS):
            lr = cfg["lr_peft"]
        else:
            lr = backbone_lr
        decay = cfg["weight_decay"] if (p.ndim >= 2 and "crf." not in name) else 0.0
        buckets[(lr, decay)].append(p)
    return [{"params": params, "lr": lr, "base_lr": lr, "weight_decay": decay} for (lr, decay), params in buckets.items()]

def lr_factor(step, total_steps, warmup_frac):
    warmup = max(1, int(total_steps * warmup_frac))
    if step < warmup:
        return (step + 1) / warmup
    progress = min(1.0, (step - warmup) / max(1, total_steps - warmup))
    return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * progress))

def release(*objects):
    for obj in objects:
        del obj
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

def train_stage(cfg, stage, train_keys, eval_keys, n_epochs, signature, teacher_scores=None):
    """Train (or resume, or load) one model. stage = "dev": train on Train/, keep the epoch with the best dev
    macro-F1. stage = "final": train on Train/ + dev for a fixed number of epochs. Returns (model, info)."""
    folder = exp_dir(cfg["name"])
    last_path, model_path = os.path.join(folder, f"{stage}_last.pt"), os.path.join(folder, f"{stage}_model.pt")
    torch.manual_seed(cfg["seed"])
    model = build_model(cfg)
    params = count_parameters(model)
    print(f"[{cfg['name']} / {stage}] parameters: {params['total'] / 1e6:.2f}M total | trained: "
          f"{params['language_model_trainable'] / 1e6:.3f}M of the {params['language_model'] / 1e6:.2f}M language-model weights "
          f"+ {params['head'] / 1e6:.2f}M head ({params['trainable_percent']:.2f}% of all weights)")
    enc = encodings_for(cfg["backbone"], list(train_keys) + list(eval_keys), cfg["max_sent_tokens"])
    n_sentences = sum(len(DOCS[k]["text"]) for k in train_keys)
    steps_per_epoch = math.ceil(len(training_chunks(train_keys, cfg["chunk_sentences"], 0)) / cfg["chunks_per_batch"])
    total_steps = steps_per_epoch * cfg["epochs"]
    optimizer = torch.optim.AdamW(optimizer_groups(model, cfg), betas=(0.9, 0.98))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    class_weights = None
    if cfg["class_weighting"] and cfg["decoder"] == "softmax":
        counts = np.bincount([l for k in train_keys for l in DOCS[k]["labels"]], minlength=NUM_LABELS).astype(np.float64)
        weights = 1.0 / np.sqrt(counts + 10.0)
        class_weights = torch.tensor(weights / weights.mean(), dtype=torch.float32, device=DEVICE)

    state = {"epoch": 0, "history": [], "best_f1": -1.0, "best_epoch": 0, "best_metrics": None, "bad_epochs": 0, "seconds": 0.0,
             "peak_memory_mb": 0.0}
    if os.path.exists(last_path):
        saved = torch_load(last_path)
        if saved.get("signature") == signature:
            load_trainable(model, saved["model"])
            optimizer.load_state_dict(saved["optimizer"])
            scaler.load_state_dict(saved["scaler"])
            state = saved["state"]
            print(f"[{cfg['name']} / {stage}] resuming after epoch {state['epoch']}")
        del saved
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    stopped_early = False
    for epoch in range(state["epoch"], n_epochs):
        if eval_keys and state["bad_epochs"] >= cfg["patience"]:
            stopped_early = True
            break
        t0 = time.time()
        model.train()
        order = training_chunks(train_keys, cfg["chunk_sentences"], cfg["seed"] * 1000 + epoch)
        random.Random(cfg["seed"] * 1000 + epoch).shuffle(order)
        torch.manual_seed(cfg["seed"] * 1000 + epoch)
        loss_sum, hard_sum, soft_sum, n_batches = 0.0, 0.0, 0.0, 0
        size = cfg["chunks_per_batch"]
        bar = tqdm(range(0, len(order), size), desc=f"{cfg['name']} {stage} epoch {epoch + 1}/{n_epochs}", leave=False)
        for b, i in enumerate(bar):
            step = epoch * steps_per_epoch + b
            factor = lr_factor(step, total_steps, cfg["warmup_frac"])
            for group in optimizer.param_groups:
                group["lr"] = group["base_lr"] * factor
            scores, tags, mask, teacher = chunk_forward(model, order[i:i + size], enc, teacher_scores)
            loss, hard, soft = model.loss(scores.float(), tags, mask, cfg, class_weights, teacher)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], cfg["grad_clip"])
            scaler.step(optimizer)
            scaler.update()
            loss_sum += loss.item()
            hard_sum += hard.item()
            soft_sum += soft.item() if soft is not None else 0.0
            n_batches += 1
            if b % 10 == 0:
                bar.set_postfix(loss=f"{loss_sum / n_batches:.4f}")
        record = {"epoch": epoch + 1, "train_loss": loss_sum / max(1, n_batches), "label_loss": hard_sum / max(1, n_batches)}
        if teacher_scores is not None:
            record["distillation_loss"] = soft_sum / max(1, n_batches)
        state["epoch"] = epoch + 1
        message = f"[{cfg['name']} / {stage}] epoch {epoch + 1:2d}/{n_epochs} | train loss {record['train_loss']:.4f}"
        if eval_keys:
            metrics, _ = evaluate(model, cfg, eval_keys, enc, desc="validation")
            record.update(dev_macro_f1=metrics["macro_f1"], dev_accuracy=metrics["accuracy"],
                          **{f"dev_macro_f1_{d}": metrics[f"macro_f1_{d}"] for d in DOMAINS if f"macro_f1_{d}" in metrics})
            message += (f" | dev macro-F1 {metrics['macro_f1']:.4f} (P {metrics['macro_precision']:.4f}, R {metrics['macro_recall']:.4f})"
                        + "".join(f" | {d} {metrics[f'macro_f1_{d}']:.4f}" for d in DOMAINS if f"macro_f1_{d}" in metrics)
                        + f" | accuracy {metrics['accuracy']:.4f}")
            if metrics["macro_f1"] > state["best_f1"]:
                state.update(best_f1=metrics["macro_f1"], best_epoch=epoch + 1, best_metrics=metrics, bad_epochs=0)
                save_atomic({"model": trainable_state(model), "signature": signature, "epoch": epoch + 1}, model_path)
                message += "  <- best"
            else:
                state["bad_epochs"] += 1
        message += f" | {time.time() - t0:.0f} s"
        state["history"].append(record)
        state["seconds"] += time.time() - t0
        if DEVICE.type == "cuda":
            state["peak_memory_mb"] = max(state["peak_memory_mb"], torch.cuda.max_memory_allocated() / 1e6)
        save_atomic({"model": trainable_state(model), "optimizer": optimizer.state_dict(), "scaler": scaler.state_dict(),
                     "state": state, "signature": signature}, last_path)
        print(message, flush=True)

    if eval_keys:
        if stopped_early or state["bad_epochs"] >= cfg["patience"]:
            print(f"[{cfg['name']} / {stage}] early stop: no improvement for {cfg['patience']} epochs.")
        load_trainable(model, torch_load(model_path)["model"])
    else:
        save_atomic({"model": trainable_state(model), "signature": signature, "epoch": state["epoch"]}, model_path)
    if os.path.exists(last_path):
        os.remove(last_path)
    info = {"best_epoch": state["best_epoch"] or state["epoch"], "history": state["history"], "metrics": state["best_metrics"],
            "train_seconds": state["seconds"], "peak_memory_mb": state["peak_memory_mb"], "parameters": params,
            "train_sentences": n_sentences}
    return model, info

def load_stage_model(cfg, stage):
    model = build_model(cfg, verbose=False)
    load_trainable(model, torch_load(os.path.join(exp_dir(cfg["name"]), f"{stage}_model.pt"))["model"])
    return model

## 10. Run the experiments

For each experiment: train (best epoch on dev) → test predictions → `submission.zip`. Finished experiments are loaded from disk. One failing experiment does not stop the others.

In [10]:
RESULTS = {}

def teacher_scores_for(cfg, stage, keys):
    """Role scores of the teacher model for the student's training documents (knowledge distillation). In the dev
    stage the teacher's dev model is used, so the dev documents stay unseen by both."""
    teacher_cfg = experiment_config(cfg["teacher"])
    teacher_stage = stage if os.path.exists(os.path.join(exp_dir(teacher_cfg["name"]), f"{stage}_model.pt")) else "dev"
    teacher = load_stage_model(teacher_cfg, teacher_stage)
    enc = encodings_for(teacher_cfg["backbone"], keys, teacher_cfg["max_sent_tokens"])
    scores = predict_scores(teacher, teacher_cfg, keys, enc, desc="teacher scores")
    print(f"[{cfg['name']} / {stage}] distilling from {teacher_cfg['name']} ({teacher_stage} model, "
          f"{count_parameters(teacher)['total'] / 1e6:.1f}M parameters)")
    release(teacher)
    return scores

def write_predictions(folder, predicted):
    """prediction.json: {doc_id: [role, role, ...]} with exactly one role per test sentence, in the original order,
    zipped as submission.zip. Everything is checked before the file is written."""
    output = {}
    for key in SPLIT_KEYS["test"]:
        roles = [LABELS[i] for i in predicted[key]]
        if len(roles) != len(DOCS[key]["text"]):
            raise RuntimeError(f"{key}: {len(roles)} predictions for {len(DOCS[key]['text'])} sentences")
        output[key[2]] = roles
    prediction_path, zip_path = os.path.join(folder, "prediction.json"), os.path.join(folder, "submission.zip")
    with open(prediction_path, "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(prediction_path, arcname="prediction.json")
    with zipfile.ZipFile(zip_path, "r") as z:
        check = json.loads(z.read("prediction.json").decode("utf-8"))
        assert z.namelist() == ["prediction.json"]
    assert set(check) == {k[2] for k in SPLIT_KEYS["test"]}, "a test document is missing"
    for key in SPLIT_KEYS["test"]:
        assert len(check[key[2]]) == len(DOCS[key]["text"]) and all(r in LABEL_TO_ID for r in check[key[2]])
    return zip_path, sum(len(v) for v in output.values())

def run_experiment(name):
    """Dev stage -> (optional final stage) -> test predictions. Every finished part is loaded from disk."""
    if name in RESULTS:
        return RESULTS[name]
    cfg = experiment_config(name)
    if cfg["backbone"] not in PHASE1:
        print(f"[{name}] SKIPPED: no Phase-1 checkpoint for model {cfg['backbone']} is attached.")
        return None
    teacher_signature = ""
    if cfg["teacher"]:
        teacher_result = run_experiment(cfg["teacher"])
        if teacher_result is None:
            print(f"[{name}] SKIPPED: its teacher {cfg['teacher']} could not be run.")
            return None
        teacher_signature = teacher_result["signature"]
    folder = exp_dir(name)
    dev_signature = config_signature(cfg, "dev" + teacher_signature)
    signature = config_signature(cfg, FINAL_MODE + teacher_signature)
    results_path, dev_path = os.path.join(folder, "results.json"), os.path.join(folder, "dev_results.json")
    if os.path.exists(results_path) and os.path.exists(os.path.join(folder, "submission.zip")):
        saved = load_json_file(results_path)
        if saved.get("signature") == signature:
            print(f"[{name}] already finished: dev macro-F1 {saved['dev']['macro_f1']:.4f} (loaded from disk)")
            RESULTS[name] = saved
            return saved

    train_keys = [k for k in SPLIT_KEYS["train"] if k[1] in cfg["train_domains"]]
    dev_keys = [k for k in SPLIT_KEYS["dev"] if k[1] in cfg["train_domains"]]
    all_dev_keys = SPLIT_KEYS["dev"]
    model = None
    dev = load_json_file(dev_path) if os.path.exists(dev_path) else None
    if dev is None or dev.get("signature") != dev_signature or not os.path.exists(os.path.join(folder, "dev_model.pt")):
        teacher = teacher_scores_for(cfg, "dev", train_keys) if cfg["teacher"] else None
        model, info = train_stage(cfg, "dev", train_keys, dev_keys, cfg["epochs"], dev_signature, teacher)
        enc = encodings_for(cfg["backbone"], all_dev_keys, cfg["max_sent_tokens"])
        metrics, predicted = evaluate(model, cfg, all_dev_keys, enc, "dev evaluation")
        gold = {k: DOCS[k]["labels"] for k in all_dev_keys}
        dev = {"signature": dev_signature, "best_epoch": info["best_epoch"], "history": info["history"],
               **{k: v for k, v in metrics.items()}, "confusions": confusions(gold, predicted, all_dev_keys),
               "parameters": info["parameters"], "train_seconds": info["train_seconds"],
               "peak_memory_mb": info["peak_memory_mb"], "train_sentences": info["train_sentences"]}
        write_json(dev, dev_path)
        write_json({k[2]: [LABELS[i] for i in predicted[k]] for k in all_dev_keys}, os.path.join(folder, "dev_predictions.json"))
    else:
        print(f"[{name}] dev stage already done: best epoch {dev['best_epoch']}, macro-F1 {dev['macro_f1']:.4f}")
    print(f"[{name}] DEV macro-F1 {dev['macro_f1']:.4f} | precision {dev['macro_precision']:.4f} | recall {dev['macro_recall']:.4f}"
          + "".join(f" | {d} {dev[f'macro_f1_{d}']:.4f}" for d in DOMAINS if f"macro_f1_{d}" in dev)
          + f" | best epoch {dev['best_epoch']}")

    final_seconds = 0.0
    if FINAL_MODE == "retrain_all":
        release(model)
        all_keys = train_keys + dev_keys
        final_path = os.path.join(folder, "final_model.pt")
        if os.path.exists(final_path) and torch_load(final_path).get("signature") == signature:
            print(f"[{name}] final model already trained; loading it.")
            model = load_stage_model(cfg, "final")
        else:
            teacher = teacher_scores_for(cfg, "final", all_keys) if cfg["teacher"] else None
            model, info = train_stage(cfg, "final", all_keys, [], dev["best_epoch"], signature, teacher)
            final_seconds = info["train_seconds"]
    elif model is None:
        model = load_stage_model(cfg, "dev")

    test_keys = SPLIT_KEYS["test"]
    enc = encodings_for(cfg["backbone"], test_keys, cfg["max_sent_tokens"])
    t0 = time.time()
    predicted = decode_all(model, predict_scores(model, cfg, test_keys, enc, "test prediction"))
    inference_seconds = time.time() - t0
    zip_path, n_sentences = write_predictions(folder, predicted)
    result = {"name": name, "signature": signature, "config": {k: v for k, v in cfg.items() if k != "name"},
              "phase1_checkpoint": PHASE1[cfg["backbone"]]["dir"], "final_mode": FINAL_MODE, "dev": dev,
              "final_train_seconds": final_seconds, "test_sentences": n_sentences,
              "test_role_counts": dict(Counter(LABELS[i] for k in test_keys for i in predicted[k])),
              "inference_seconds": inference_seconds, "sentences_per_second": n_sentences / max(1e-9, inference_seconds),
              "submission": zip_path}
    write_json(result, results_path)
    print(f"[{name}] test predictions: {n_sentences:,} sentences in {len(test_keys)} documents -> {zip_path}\n")
    release(model)
    RESULTS[name] = result
    return result

for _name in RUN_EXPERIMENTS:
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(experiment_config(_name)["seed"])
    try:
        run_experiment(_name)
    except Exception as _err:
        import traceback
        traceback.print_exc()
        print(f"[{_name}] FAILED: {_err} - continuing with the next experiment.\n")
        release()

[R1_D_lorma] already finished: dev macro-F1 0.5600 (loaded from disk)
[R2_B_rslora] already finished: dev macro-F1 0.5898 (loaded from disk)
[R3_C_ia3] already finished: dev macro-F1 0.1321 (loaded from disk)
[R4_A4_kd] already finished: dev macro-F1 0.5423 (loaded from disk)
Encoder check: token-embedding RMS 0.0405 vs position-encoding RMS 0.7071 | last layer, cosine similarity of two DIFFERENT inputs at the same position 1.000 (neighbouring positions of one input 1.000; close to 1.0 = the output ignores the tokens)
Loaded Phase-1 model C (encoder-decoder): 6/6 layers, 48.82M parameters (156 weight tensors not needed, e.g. removed layers or the decoder)
[X8_C_full / dev] parameters: 50.93M total | trained: 48.819M of the 48.82M language-model weights + 2.11M head (100.00% of all weights)


tokenizing for model C:   0%|          | 0/90 [00:00<?, ?it/s]

X8_C_full dev epoch 1/12:   0%|          | 0/150 [00:00<?, ?it/s]

validation:   0%|          | 0/17 [00:00<?, ?it/s]

[X8_C_full / dev] epoch  1/12 | train loss 2.1465 | dev macro-F1 0.0322 (P 0.0492, R 0.0914) | CL 0.0317 | IT 0.0377 | accuracy 0.2082  <- best | 39 s


X8_C_full dev epoch 2/12:   0%|          | 0/150 [00:00<?, ?it/s]

... (115 outputs left out)

dev evaluation:   0%|          | 0/17 [00:00<?, ?it/s]

[X3_D_lorma_balanced] DEV macro-F1 0.5646 | precision 0.5927 | recall 0.5601 | CL 0.5024 | IT 0.5482 | best epoch 9


test prediction:   0%|          | 0/18 [00:00<?, ?it/s]

[X3_D_lorma_balanced] test predictions: 2,197 sentences in 10 documents -> /kaggle/working/cs781_phase3/X3_D_lorma_balanced/submission.zip



## 11. Results and analysis

In [11]:
finished = {name: r for name, r in RESULTS.items() if r is not None}
for _name in EXPERIMENTS:
    _path = os.path.join(WORK_ROOT, _name, "results.json")
    if _name not in finished and os.path.exists(_path):
        finished[_name] = load_json_file(_path)

rows = []
for name, r in sorted(finished.items(), key=lambda kv: -kv[1]["dev"]["macro_f1"]):
    cfg, dev = r["config"], r["dev"]
    rows.append({"experiment": name, "model": cfg["backbone"] + (f"[:{cfg['n_layers']}]" if cfg["n_layers"] else ""),
                 "technique": cfg["technique"] + ("-rs" if cfg.get("lora_scaling") == "rs" else "") + (" + KD" if cfg["teacher"] else ""),
                 "context": cfg["context"], "decoder": cfg["decoder"] + (" bal" if cfg["class_weighting"] else ""),
                 "train_on": "+".join(cfg["train_domains"]),
                 "dev_macro_f1": dev["macro_f1"], "dev_macro_p": dev["macro_precision"], "dev_macro_r": dev["macro_recall"],
                 "dev_f1_CL": dev.get("macro_f1_CL", float("nan")), "dev_f1_IT": dev.get("macro_f1_IT", float("nan")),
                 "dev_accuracy": dev["accuracy"], "best_epoch": dev["best_epoch"],
                 "params_M": dev["parameters"]["total"] / 1e6, "trainable_M": dev["parameters"]["trainable"] / 1e6,
                 "lm_trained_M": dev["parameters"].get("language_model_trainable", 0) / 1e6,
                 "trainable_%": dev["parameters"]["trainable_percent"], "train_min": dev["train_seconds"] / 60,
                 "peak_mem_GB": dev["peak_memory_mb"] / 1000, "sentences_per_s": r["sentences_per_second"]})

if not rows:
    print("No finished experiments yet.")
else:
    header = (f"{'experiment':26s} {'model':6s} {'technique':12s} {'context':7s} {'decoder':11s} {'macroF1':>8s} {'P':>6s} {'R':>6s} "
              f"{'CL':>6s} {'IT':>6s} {'acc':>6s} {'params':>7s} {'trained':>8s} {'%':>6s} {'train':>6s} {'mem':>5s} {'sent/s':>7s}")
    print("Dev set (CL + IT dev documents). macro-F1 over the roles that occur; CL / IT = macro-F1 per domain; "
          "'trained' = all trained weights (language model + head).\n" + header + "\n" + "-" * len(header))
    for x in rows:
        print(f"{x['experiment']:26s} {x['model']:6s} {x['technique']:12s} {x['context']:7s} {x['decoder']:11s} "
              f"{x['dev_macro_f1']:8.4f} {x['dev_macro_p']:6.3f} {x['dev_macro_r']:6.3f} {x['dev_f1_CL']:6.3f} {x['dev_f1_IT']:6.3f} "
              f"{x['dev_accuracy']:6.3f} {x['params_M']:6.1f}M {x['trainable_M']:7.3f}M {x['trainable_%']:6.2f} "
              f"{x['train_min']:5.1f}m {x['peak_mem_GB']:4.1f}G {x['sentences_per_s']:7.0f}")
    with open(os.path.join(WORK_ROOT, "results_summary.csv"), "w", encoding="utf-8") as f:
        f.write(",".join(rows[0].keys()) + "\n")
        for x in rows:
            f.write(",".join(str(round(v, 5)) if isinstance(v, float) else str(v) for v in x.values()) + "\n")

    names = [x["experiment"] for x in rows]
    print("\nF1 per role (dev):")
    print(f"{'role':24s} {'gold':>5s} " + " ".join(f"{n[:13]:>13s}" for n in names))
    for lab in LABELS:
        per = [finished[n]["dev"]["per_label"].get(lab) for n in names]
        gold_n = next((p["gold"] for p in per if p), 0)
        print(f"{lab:24s} {gold_n:5d} " + " ".join(f"{p['f1']:13.4f}" if p else f"{'-':>13s}" for p in per))
    print(f"\nMost frequent confusions of {names[0]} (gold -> predicted, sentences):")
    for a, b, n in finished[names[0]]["dev"]["confusions"]:
        print(f"  {a:22s} -> {b:22s} {n:5d}")
    print(f"\nSummary table saved to {os.path.join(WORK_ROOT, 'results_summary.csv')}")

Dev set (CL + IT dev documents). macro-F1 over the roles that occur; CL / IT = macro-F1 per domain; 'trained' = all trained weights (language model + head).
experiment                 model  technique    context decoder      macroF1      P      R     CL     IT    acc  params  trained      %  train   mem  sent/s
-----------------------------------------------------------------------------------------------------------------------------------------------------------
R2_B_rslora                B      lora-rs      bilstm  crf           0.5898  0.688  0.560  0.481  0.568  0.653   68.5M   2.439M   3.56  12.1m  4.1G    1065
X3_D_lorma_balanced        D      lorma        bilstm  softmax bal   0.5646  0.593  0.560  0.502  0.548  0.634   67.1M   2.504M   3.73   9.3m  3.3G    1160
R1_D_lorma                 D      lorma        bilstm  crf           0.5600  0.619  0.541  0.465  0.535  0.641   67.1M   2.504M   3.73   9.6m  3.3G    1210
R4_A4_kd                   A[:4]  full + KD    bilstm  crf     

## 12. Submission files

In [12]:
if rows:
    print("Submission files (best dev score first):")
    for x in rows:
        print(f"  {x['experiment']:26s} dev macro-F1 {x['dev_macro_f1']:.4f}  {finished[x['experiment']]['submission']}")
    best_zip = os.path.join(WORK_ROOT, rows[0]["experiment"], "submission.zip")
    if os.path.exists(best_zip):
        shutil.copy2(best_zip, os.path.join(WORK_ROOT, "submission.zip"))
        print(f"\nBest on dev: {rows[0]['experiment']} -> copied to {os.path.join(WORK_ROOT, 'submission.zip')}")

Submission files (best dev score first):
  R2_B_rslora                dev macro-F1 0.5898  /kaggle/working/cs781_phase3/R2_B_rslora/submission.zip
  X3_D_lorma_balanced        dev macro-F1 0.5646  /kaggle/working/cs781_phase3/X3_D_lorma_balanced/submission.zip
  R1_D_lorma                 dev macro-F1 0.5600  /kaggle/working/cs781_phase3/R1_D_lorma/submission.zip
  R4_A4_kd                   dev macro-F1 0.5423  /kaggle/working/cs781_phase3/R4_A4_kd/submission.zip
  X2_D_full                  dev macro-F1 0.5255  /kaggle/working/cs781_phase3/X2_D_full/submission.zip
  X8_C_full                  dev macro-F1 0.1516  /kaggle/working/cs781_phase3/X8_C_full/submission.zip
  R3_C_ia3                   dev macro-F1 0.1321  /kaggle/working/cs781_phase3/R3_C_ia3/submission.zip

Best on dev: R2_B_rslora -> copied to /kaggle/working/cs781_phase3/submission.zip
